# Experiment 1: Temperature as an Input Feature

Companion notebook to [`README.md`](README.md) for the DAAN 570 course project (Temperature Prediction with Deep Learning).

**Hypothesis**: `temperature_2m` is the prediction target, but its own recent history is also a strong autoregressive signal. Does including the past `WINDOW_SIZE` hours of temperature alongside the existing weather covariates improve test MAE/RMSE over the baseline (weather-only) Transformer?

**What changed vs. the baseline** (`src/models/transformer/transformer_encoder_experiment.ipynb`): nothing except the input features -- see [`config.py`](config.py). Architecture, hyperparameters, data splits, and training loop are all shared with the baseline via `src/models/transformer/architecture.py` and `src/models/transformer/training.py`.

This notebook can run either locally (if `data/splits/` is already on your machine) or on Google Colab with a GPU runtime. The first code cell below handles both cases automatically.

## Setup

**Running on Colab**: this cell clones the (private) repo, installs dependencies, and pulls the Git-LFS-tracked raw data. One-time setup: create a fine-grained, read-only GitHub PAT for this repo and store it in Colab's Secrets manager (key icon in the left sidebar) under the name `GITHUB_PAT`, then grant this notebook access to it when prompted. Also select **Runtime > Change runtime type > GPU** before running.

**Running locally**: this cell detects that it's not on Colab and does nothing -- it assumes you already have the repo cloned, dependencies installed (`pip install -r requirements.txt`), and `data/splits/` available (or `data/raw/` present, from which splits regenerate automatically).

In [ ]:
import shutil
import sys
from pathlib import Path

BRANCH = "exp01_temperature_feature"


def run(cmd):
    exit_code = get_ipython().system(cmd)
    if exit_code != 0:
        raise RuntimeError(f"Command failed (exit {exit_code}): {cmd}")


try:
    import google.colab
    from google.colab import userdata

    REPO_DIR = Path("/content/repo")
    token = userdata.get("GITHUB_PAT")
    repo_url = f"https://{token}@github.com/LarryGreen-alt/Temperature_Prediction_DeepLearning.git"

    # A directory can be left behind by a previous failed attempt without
    # being a real git checkout -- only trust it if .git is actually there.
    if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
        shutil.rmtree(REPO_DIR)

    if (REPO_DIR / ".git").is_dir():
        run(f"git -C {REPO_DIR} fetch -q origin {BRANCH}")
        run(f"git -C {REPO_DIR} checkout -q {BRANCH}")
        run(f"git -C {REPO_DIR} pull -q origin {BRANCH}")
    else:
        run(f"git clone -q -b {BRANCH} {repo_url} {REPO_DIR}")

    get_ipython().run_line_magic("cd", str(REPO_DIR))
    run("pip install -q -r requirements.txt")
    run("apt-get -qq install -y git-lfs")
    run("git lfs install")
    run("git lfs pull")

    PROJECT_ROOT = REPO_DIR
except ImportError:

    def find_project_root(marker="weather_main.py"):
        for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
            if (candidate / marker).exists():
                return candidate
        raise FileNotFoundError(f"Could not find project root (looking for {marker})")

    PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import pandas as pd

from src.models.common import data
from src.models.common.compare import find_cached_experiment, compare
from src.models.common.plotting import plot_loss_curve, plot_mae_curve, plot_prediction_curve
from src.models.transformer.training import run_training, load_cached_results, timestamp_now
from src.models.transformer.experiments.exp01_temperature_feature.config import (
    CONFIG, EXPERIMENT_NAME, FEATURE_COLUMNS
)

## Data

Same source as the baseline: `data/splits/{train,dev,test}.csv`, derived from `data/raw/*.csv` via `src/data/preprocess.py` -> `src/data/split_dataset.py` (chronological 70/15/15 split per city). `data.load_splits()` regenerates `data/splits/` automatically if missing.

In [ ]:
train_df, dev_df, test_df = data.load_splits()

print("Train rows:", len(train_df))
print("Dev rows  :", len(dev_df))
print("Test rows :", len(test_df))
train_df.head()

## Configuration

Hyperparameters (`CONFIG`) are identical to the baseline Transformer run -- the only difference from baseline is `FEATURE_COLUMNS`, which adds `temperature_2m` to the baseline's weather covariates.

In [ ]:
print(CONFIG)
print()
print("Feature columns:", FEATURE_COLUMNS)
print("Baseline had:   ", data.FEATURE_COLUMNS)

## Train or load

Same caching behavior as the baseline notebook: this experiment's runs live under their own `models/Transformer/exp01_temperature_feature/` directory (separate from baseline), so an exact `CONFIG` match here only ever compares against other exp01 runs. Set `FORCE_RETRAIN = True` to bypass the cache and retrain regardless.

In [ ]:
FORCE_RETRAIN = False

MODEL_ROOT = data.PROJECT_ROOT / "models" / "Transformer" / EXPERIMENT_NAME
checkpoint_path = MODEL_ROOT / "checkpoints" / "best.keras"

config_dict = CONFIG.to_dict()
cached_dir = None if FORCE_RETRAIN else find_cached_experiment(f"Transformer/{EXPERIMENT_NAME}", config_dict)

if cached_dir:
    print(f"Found a cached experiment matching this config: {cached_dir.name}")
    results = load_cached_results(cached_dir)
else:
    print("No cached experiment matches this config (or FORCE_RETRAIN=True) -- training now...")
    experiment_dir = MODEL_ROOT / "experiments" / timestamp_now()
    results = run_training(
        CONFIG, experiment_dir, checkpoint_path,
        feature_columns=FEATURE_COLUMNS, show_plots=True
    )

print("Experiment dir:", results["experiment_dir"])

## Results

In [ ]:
print(f"Test Loss : {results['loss']:.4f}")
print(f"Test MAE  : {results['mae']:.4f}")
print(f"Test RMSE : {results['rmse']:.4f}")
print(f"Epochs trained: {results['epochs_trained']}")

figure_dir = results["experiment_dir"] / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

plot_loss_curve(results["history"], figure_dir, show=True)
plot_mae_curve(results["history"], figure_dir, show=True)
plot_prediction_curve(results["y_test"], results["predictions"], figure_dir, show=True)

pd.read_csv(results["experiment_dir"] / "predictions.csv").head(10)

## Comparison to the baseline Transformer

Reuses `src/models/common/compare.py`'s `compare()`, now generalized to accept an explicit list of model names so it can look inside this experiment's namespaced results directory alongside the baseline.

In [ ]:
compare(["Transformer", f"Transformer/{EXPERIMENT_NAME}"])

## Save results

**If running on Colab**, run the cell below to zip this experiment's output folder together with this executed notebook (so the professor can see the real run, including these plots and outputs) and download it as one file. Unzip it directly into your local repo checkout, review with `git status`/`git diff`, then commit and push from your own machine.

**If running locally**, your results are already sitting in the repo at `results["experiment_dir"]` -- nothing more to do.

In [ ]:
try:
    import google.colab
    from google.colab import files

    zip_name = f"{EXPERIMENT_NAME}_results.zip"
    notebook_path = f"src/models/transformer/experiments/{EXPERIMENT_NAME}/experiment.ipynb"
    results_glob = f"models/Transformer/{EXPERIMENT_NAME}"

    get_ipython().system(f"zip -r {zip_name} {results_glob} {notebook_path}")
    files.download(zip_name)
except ImportError:
    print("Not running on Colab -- results are already in your local repo checkout.")